# Pertemuan 06 — Regresi Linear (Bagian 1)

Pada modul ini kita akan mempelajari **perumusan regresi linear**, **fungsi basis**, dan **tafsiran geometris** dari regresi linear.

> **Perubahan jenis persoalan.** Tiga pertemuan terakhir membahas *unsupervised learning* (k-means, GMM), yaitu mengelompokkan data tanpa label. Mulai sekarang kita kembali ke *supervised learning*, tetapi dengan target berupa **angka kontinu**, bukan kategori. Inilah yang disebut **regresi**.

**Prasyarat:** aljabar linear dasar (vektor, perkalian matriks, proyeksi) dan konsep overfitting dari Pertemuan 03.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

np.random.seed(42)


## Tujuan Pembelajaran

Notebook ini dirancang sebagai penjelajahan interaktif atas regresi linear.

**1. Memahami dasar regresi linear**
- Menelusuri gagasan mencocokkan sebuah garis lurus pada sekumpulan titik data.
- Mengenali parameter model linear: **intersep** (titik potong) dan **kemiringan** (*slope*).

**2. Menelusuri fungsi basis (basis function)**
- Mempelajari bagaimana berbagai fungsi basis (polinomial, Gaussian, Fourier) mengubah ruang masukan.
- Memahami bagaimana transformasi itu memungkinkan regresi linear memodelkan hubungan **non-linear**.

**3. Membandingkan regresi linear dan polinomial**
- Melatih dan membandingkan kedua model pada data yang sama.
- Memahami pertukaran (*trade-off*) antara *underfitting* dan *overfitting*.

**4. Memvisualkan geometri regresi linear**
- Memahami tafsiran geometris regresi linear dalam ruang dua dan tiga dimensi.
- Menelusuri konsep **residual**, **hiperbidang** (*hyperplane*), dan **rentang** (*span*) dari fitur masukan.

**5. Menganalisis residual**
- Mempelajari cara membaca plot residual untuk menilai kecocokan model.
- Memahami konsep **heteroskedastisitas** dan implikasinya.

**6. Menerapkan regresi linear pada data nyata**
- Memakai dataset California Housing untuk melatih model regresi linear sederhana.
- Mengevaluasi model dengan metrik **Mean Squared Error (MSE)** dan **R-kuadrat**.

> **Cara memakai modul ini.** Jalankan sel kode berurutan dari atas ke bawah. Setiap bagian diawali penjelasan dan diikuti sel visualisasi.


---
## 1. Dasar-dasar Regresi Linear

Model regresi linear paling dasar memprediksi keluaran $y$ sebagai **kombinasi linear** dari fitur masukan $x$ dan parameter model $w$. Bentuk umumnya:

$$
y(x, w) = x^T w
$$

Di sini $x$ adalah vektor masukan, $w$ adalah vektor parameter, dan $y(x, w)$ adalah keluaran yang diprediksi. Untuk data dengan $D$ fitur, model ini dapat dituliskan sebagai:

$$
y(x, w) = w_0 + w_1 x_1 + w_2 x_2 + \dots + w_D x_D
$$

dengan:

| Simbol | Arti |
|---|---|
| $w_0$ | **Intersep** (*bias*), yaitu nilai $y$ ketika seluruh fitur bernilai nol |
| $w_1, \dots, w_D$ | **Bobot** (koefisien) bagi setiap fitur masukan |
| $x = (x_1, \dots, x_D)^T \in \mathbb{R}^D$ | Vektor masukan |
| $t$ | Nilai target yang sebenarnya |

**Tujuan regresi linear** adalah mempelajari parameter $w$ dari data latih sedemikian rupa sehingga prediksi $y(x, w)$ sedekat mungkin dengan nilai target sebenarnya $t$.

> **Mengapa disebut "linear"?** Bukan karena grafiknya harus berupa garis lurus, melainkan karena modelnya **linear terhadap parameter $w$**. Nanti pada Bagian 2 kita akan melihat bahwa dengan fungsi basis, model ini bisa menghasilkan kurva melengkung — dan tetap disebut regresi linear.

Pada sel di bawah kita membuat data buatan dari hubungan $t = 2x + 1$ ditambah derau acak, lalu menggambar garis sebenarnya sebagai pembanding.


In [ ]:
# Membuat data buatan: t = 2x + 1 + derau acak
N = 50
x = np.random.rand(N) * 10
t = 2 * x + 1 + np.random.randn(N) * 2

x_pred = np.linspace(0, 10, 100)
y_conceptual = 2 * x_pred + 1

plt.figure(figsize=(8, 6))
plt.scatter(x, t, alpha=0.7, label='Data $(x_n, t_n)$', s=50)
plt.plot(x_pred, y_conceptual, color='red', linestyle='--', label='Garis prediksi y(x)', linewidth=4.0)
plt.xlabel('x (masukan)', fontsize=20)
plt.ylabel('Keluaran (t atau y(x))', fontsize=20)
plt.title('Data dan Fungsi Regresi Linear', fontsize=20)
plt.legend(fontsize=15)
plt.grid(True)
# plt.savefig('conceptual_prediction.png')
plt.show()

Garis merah putus-putus adalah hubungan yang **sebenarnya**, yaitu $y = 2x + 1$. Titik-titik biru menyimpang dari garis itu karena adanya derau acak.

Dalam kasus nyata kita tidak pernah tahu garis sebenarnya — yang kita punya hanya titik-titik birunya. Tugas regresi linear adalah **menemukan kembali** garis itu dari data yang teramati.

In [ ]:
example_w0 = 1.0
example_w1 = 2.0

x_pred = np.linspace(0, 10, 100)
y_example_linear = example_w0 + example_w1 * x_pred

plt.figure(figsize=(8, 6))
plt.scatter(x, t, alpha=0.7, label='Data $(x_n, t_n)$', s=100)
plt.plot(x_pred, y_example_linear, color='red', linestyle='--', label=f'Prediksi linear $y(x) = {example_w0} + {example_w1}x$', linewidth=4.0)

plt.axhline(0, color='grey', lw=0.5)
plt.axvline(0, color='grey', lw=0.5)

plt.plot(0, example_w0, 'go', markersize=12, label=f'Intersep ($w_0={example_w0}$)')

x_point_1 = 2
x_point_2 = 4
y_point_1 = example_w0 + example_w1 * x_point_1
y_point_2 = example_w0 + example_w1 * x_point_2

plt.plot([x_point_1, x_point_2], [y_point_1, y_point_1], 'k--', linewidth=4.0)
plt.plot([x_point_2, x_point_2], [y_point_1, y_point_2], 'k--', linewidth=4.0)
plt.text(x_point_1 + (x_point_2 - x_point_1)/2, y_point_1 - 2, rf'$\Delta x = {x_point_2 - x_point_1}$', horizontalalignment='center', verticalalignment='top', fontsize=15)
plt.text(x_point_2 + 0.3, y_point_1 + (y_point_2 - y_point_1)/2, rf'$\Delta y = {y_point_2 - y_point_1}$', horizontalalignment='left', verticalalignment='center', fontsize=15)
plt.text(8, 2, rf'Kemiringan $w_1 = \frac{{\Delta y}}{{\Delta x}} = {example_w1}$', horizontalalignment='right', verticalalignment='bottom', fontsize=15)

plt.xlabel('x (masukan)', fontsize=20)
plt.ylabel('Keluaran (t atau y(x))', fontsize=20)
plt.title('Intersep dan Kemiringan pada Fungsi Regresi Linear', fontsize=20)
plt.legend(fontsize=15)
plt.grid(True)
# plt.savefig('intercept_slope_illustration.png')
plt.show()

Grafik ini memperjelas arti kedua parameter model:

* **Intersep $w_0$** (titik hijau) adalah nilai $y$ saat $x = 0$, yaitu tempat garis memotong sumbu tegak.
* **Kemiringan $w_1$** adalah $\Delta y / \Delta x$, yaitu besar perubahan $y$ untuk setiap penambahan satu satuan $x$.

Pada contoh ini $w_1 = 2$, artinya setiap kenaikan $x$ sebesar 1 satuan diikuti kenaikan $y$ sebesar 2 satuan.

### 1.1 Regresi Linear dalam Dimensi Lebih Tinggi

Pada dimensi yang lebih tinggi, nilai-nilai prediksi dari model regresi linear membentuk sebuah **hiperbidang** (*hyperplane*) dalam $\mathbb{R}^{D+1}$, dengan $D$ adalah banyaknya fitur masukan.

**Bagaimana membayangkannya:**

| Jumlah fitur | Bentuk prediksinya |
|---|---|
| 1 fitur | Garis lurus pada bidang dua dimensi |
| 2 fitur | Bidang datar dalam ruang tiga dimensi |
| $D$ fitur | Hiperbidang dalam ruang $(D+1)$ dimensi |

Hiperbidang ini mewakili hubungan antara fitur masukan dan keluaran yang diprediksi. Setiap titik pada hiperbidang bersesuaian dengan nilai prediksi bagi satu kombinasi fitur tertentu. Bentuk geometri hiperbidang itu ditentukan oleh koefisien model (kemiringan) dan intersepnya.

Sel berikut membangkitkan data dengan **dua fitur** ($x_1$ dan $x_2$), melatih model regresi linear, lalu menggambar bidang hasil pelatihannya dalam ruang tiga dimensi. Putar-putar grafiknya dengan tetikus untuk melihat dari berbagai sudut.


In [ ]:
# Data buatan dengan DUA fitur: t = 2*x1 + 3*x2 + 5 + derau
N = 50
x1 = np.random.rand(N) * 10
x2 = np.random.rand(N) * 10

t = 2 * x1 + 3 * x2 + 5 + np.random.randn(N) * 5

X = np.vstack((x1, x2)).T

model = LinearRegression()
model.fit(X, t)

# Mengambil koefisien (w1, w2) dan intersep (w0) hasil pelatihan
w1, w2 = model.coef_
w0 = model.intercept_

print(f"Model hasil pelatihan: t = {w0:.2f} + {w1:.2f}*x1 + {w2:.2f}*x2")

# Membuat kisi nilai x1 dan x2 untuk menggambar hiperbidangnya
x1_surf, x2_surf = np.meshgrid(np.linspace(x1.min(), x1.max(), 12),
                               np.linspace(x2.min(), x2.max(), 12))
t_surf = w0 + w1 * x1_surf + w2 * x2_surf

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=x1, y=x2, z=t,
    mode='markers',
    marker=dict(
        size=5,
        color='blue',
        opacity=0.8
    ),
    name='Data (x₁, x₂, t)'
))

fig.add_trace(go.Surface(
    x=x1_surf, y=x2_surf, z=t_surf,
    colorscale=[[0, 'red'], [1, 'red']],
    opacity=0.5,
    showscale=False,
    name='Hiperbidang hasil pelatihan'
))

fig.update_layout(
    title='Regresi Linear Dua Fitur dan Hiperbidang Hasil Pelatihan',
    scene=dict(
        xaxis=dict(
            title='x₁',
            backgroundcolor='white',
            gridcolor='lightgray'
        ),
        yaxis=dict(
            title='x₂',
            backgroundcolor='white',
            gridcolor='lightgray'
        ),
        zaxis=dict(
            title='Keluaran (t atau y(x))',
            backgroundcolor='white',
            gridcolor='lightgray',
        ),
        bgcolor='white'
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    font=dict(size=14),
    legend=dict(font=dict(size=12)),
)

# fig.to_image(format="png", width=800, height=400)
# fig.write_html("linear_regression_3d.html")
fig.show()

Titik-titik biru adalah data, sedangkan bidang merah adalah hiperbidang hasil pelatihan. Perhatikan bahwa titik data berada **di atas dan di bawah** bidang itu — jarak tegak setiap titik ke bidang adalah residualnya.

Bandingkan koefisien yang tercetak di atas dengan nilai sebenarnya yang kita pakai membangkitkan data ($w_0 = 5$, $w_1 = 2$, $w_2 = 3$). Nilainya tidak persis sama karena adanya derau, tetapi seharusnya cukup mendekati.

In [ ]:
# Lima titik data saja, agar residualnya mudah diamati satu per satu
x_points = np.linspace(0, 10, 5)
t_points = 2 * x_points + 1 + np.random.randn(5) * 5

def plot_polynomial_with_residuals(x_data, t_data, degree=1, color='red', label='y(x, w)'):
    w = np.polyfit(x_data, t_data, degree)
    polynomial = np.poly1d(w)

    x_plot = np.linspace(x_data.min(), x_data.max(), 100)
    y_plot = polynomial(x_plot)

    plt.figure(figsize=(10, 7))

    plt.plot(x_plot, y_plot, color=color, label=label, linewidth=4)

    plt.scatter(x_data, t_data, color='blue', label='Titik data ($t_n$)', s=150)

    # Menghitung prediksi y untuk seluruh titik
    y_predicted_points = polynomial(x_data)

    # Menghitung residual: selisih target sebenarnya dan prediksi
    residuals = t_data - y_predicted_points

    # Menggambar residual sebagai garis hijau, disertai tanda positif/negatif
    for i in range(len(x_data)):
        x_i = x_data[i]
        t_i = t_data[i]
        y_pred_i = y_predicted_points[i]
        residual_i = residuals[i]

        plt.plot([x_i, x_i], [t_i, y_pred_i], color='green', linestyle='--', linewidth=4, alpha=0.7)

        text_x_pos = x_i + (x_data.max() - x_data.min()) * 0.01
        text_y_pos = (t_i + y_pred_i) / 2

        sign = '+' if residual_i >= 0 else '-'
        plt.text(text_x_pos, text_y_pos, f'{residual_i:.2f} ({sign})',
                 fontsize=14, color='green', ha='left', va='center')

    plt.xlabel('x', fontsize=16)
    plt.ylabel('t / y(x, w)', fontsize=16)
    plt.title('Regresi Linear beserta Residualnya', fontsize=18)
    plt.legend(fontsize=14)
    plt.grid(True)
    # plt.savefig('linear_regression_with_residuals.png')
    plt.show()

plot_polynomial_with_residuals(x_points, t_points, degree=1, color='red', label='y(x, w)')

Garis hijau putus-putus adalah **residual** setiap titik, yaitu jarak tegak dari titik data ke garis prediksi. Tandanya menunjukkan arah: **positif** bila titik berada di atas garis, **negatif** bila di bawah.

> **Inilah yang diminimalkan regresi linear.** Yang dicari bukan jumlah residualnya (yang bisa saling meniadakan antara yang positif dan negatif), melainkan **jumlah kuadrat residualnya**. Mengkuadratkan membuat semua nilai menjadi positif sekaligus menghukum kesalahan besar jauh lebih berat.

---
## 2. Fungsi Basis (Basis Function)

**Fungsi basis** adalah transformasi matematis yang diterapkan pada peubah masukan agar regresi linear mampu memodelkan hubungan **non-linear**.

Gagasannya: alih-alih memakai $x$ apa adanya, kita ubah dulu $x$ menjadi bentuk lain, lalu melakukan regresi linear pada hasil transformasinya. Dengan memperluas kelas model menjadi kombinasi linear dari fungsi-fungsi non-linear yang tetap, target dapat dinyatakan sebagai:

$$
y(\mathbf{x}, \mathbf{w}) = w_0 + \sum_{j=1}^{M-1} w_j \phi_j(x)
$$

dengan:

* $\phi_j(x)$ adalah **fungsi basis** yang mentransformasi ruang masukan,
* $w_0$ adalah intersep,
* $w_j$ adalah bobot bagi setiap fungsi basis.

> **Kunci pemahamannya.** Model ini tetap **linear terhadap $w$**, meskipun $\phi_j(x)$ boleh serumit apa pun. Karena itu seluruh teori dan rumus penyelesaian regresi linear tetap berlaku. Inilah alasan fungsi basis begitu berdaya guna.

### Jenis-jenis Fungsi Basis yang Umum

**1. Basis Polinomial**

$$ \phi_j(x) = x^j $$

Contoh: $\phi_1(x) = x$, $\phi_2(x) = x^2$, $\phi_3(x) = x^3$, dan seterusnya.
Cocok untuk kurva yang melengkung mulus. Kelemahannya: bersifat **global**, sehingga mengubah data di satu ujung dapat memengaruhi bentuk kurva di ujung lain.

**2. Basis Fungsi Radial (Radial Basis Function / RBF)**

$$ \phi_j(x) = e^{-\frac{(x - \mu_j)^2}{2\sigma^2}} $$

* $\mu_j$: pusat Gaussian.
* $\sigma$: lebar (sebaran) Gaussian.

Cocok untuk pola yang **terlokalisasi** — setiap basis hanya "aktif" di sekitar pusatnya saja.

**3. Basis Sigmoid**

$$ \phi_j(x) = \sigma\left(\frac{x - \mu_j}{s}\right), \quad \sigma(a) = \frac{1}{1 + e^{-a}} $$

Berbentuk seperti tangga yang melandai. Dapat pula memakai tangen hiperbolik, karena $\tanh(a) = 2\sigma(2a) - 1$.

**4. Basis Sinusoidal (Fourier)**

$$ \phi_j(x) = \sin(x_j) \quad \text{atau} \quad \phi_j(x) = \cos(x_j) $$

Cocok untuk data yang **berkala** atau berosilasi, misalnya data musiman dan sinyal.

### Ringkasan

| Basis | Sifat | Paling cocok untuk |
|---|---|---|
| Polinomial | Global, mulus | Kurva melengkung sederhana |
| RBF / Gaussian | Lokal | Puncak atau gerombolan setempat |
| Sigmoid | Peralihan bertahap | Data dengan ambang atau titik jenuh |
| Fourier | Berkala | Pola musiman dan gelombang |

Seluruh fungsi basis ini memetakan ruang masukan ke ruang fitur berdimensi lebih tinggi, sehingga regresi linear dapat menangkap pola rumit dan non-linear. Dengan memilih fungsi basis yang tepat, model dapat disesuaikan dengan ciri khas datanya.

> **Perhatikan.** Memilih fungsi basis sama saja dengan menanamkan **bias induktif** — asumsi awal mengenai bentuk pola pada data, seperti yang dibahas pada Pertemuan 03.


In [ ]:
X_lr = np.linspace(0, 10, 100).reshape(-1, 1)
y_lr = 2 * X_lr.flatten() + np.random.normal(0, 0.5, X_lr.shape[0])

# Basis polinomial (data kuadratik berderau)
X_poly = np.linspace(0, 10, 100).reshape(-1, 1)
y_poly = 0.5 * X_poly.flatten()**2 - 3 * X_poly.flatten() + 5 + np.random.normal(0, 2, X_poly.shape[0])

# Basis Gaussian (data dengan puncak terlokalisasi)
X_gaussian = np.linspace(0, 10, 100).reshape(-1, 1)
y_gaussian = np.exp(-0.5 * (X_gaussian.flatten() - 5)**2) + np.random.normal(0, 0.1, X_gaussian.shape[0])

# Basis Fourier (data berkala)
X_fourier = np.linspace(0, 10, 100).reshape(-1, 1)
y_fourier = np.sin(2 * np.pi * X_fourier.flatten() / 5) + np.random.normal(0, 0.1, X_fourier.shape[0])  # derau diperkecil

# Melatih regresi linear biasa
model_lr = LinearRegression()
model_lr.fit(X_lr, y_lr)
y_pred_lr = model_lr.predict(X_lr)

# Melatih model dengan basis polinomial
degree = 3
model_poly = make_pipeline(PolynomialFeatures(degree), LinearRegression())
model_poly.fit(X_poly, y_poly)
y_pred_poly = model_poly.predict(X_poly)

# Gaussian Basis Function
kernel = RBF(length_scale=1.0)
model_gaussian = GaussianProcessRegressor(kernel=kernel, alpha=0.1)
model_gaussian.fit(X_gaussian, y_gaussian)
y_pred_gaussian, _ = model_gaussian.predict(X_gaussian, return_std=True)

# Fourier Basis Function
def fourier_basis(x, n_terms=4):
    basis = [np.ones_like(x)]
    for i in range(1, n_terms + 1):
        basis.append(np.sin(i * x))
        basis.append(np.cos(i * x))
    return np.column_stack(basis)

X_fourier_basis = fourier_basis(X_fourier.flatten(), n_terms=4)
model_fourier = LinearRegression()
model_fourier.fit(X_fourier_basis, y_fourier)
y_pred_fourier = model_fourier.predict(X_fourier_basis)

fig, axs = plt.subplots(2, 2, figsize=(12, 10))

# Regular Linear Regression
axs[0, 0].scatter(X_lr, y_lr, color='blue', alpha=0.5)
axs[0, 0].plot(X_lr, y_pred_lr, color='red')
axs[0, 0].set_title('A', fontsize=16, fontweight='bold')
axs[0, 0].set_xlabel('X', fontsize=14)
axs[0, 0].set_ylabel('y', fontsize=14)

# Polynomial Basis Function
axs[0, 1].scatter(X_poly, y_poly, color='blue', alpha=0.5)
axs[0, 1].plot(X_poly, y_pred_poly, color='red')
axs[0, 1].set_title('B', fontsize=16, fontweight='bold')
axs[0, 1].set_xlabel('X', fontsize=14)
axs[0, 1].set_ylabel('y', fontsize=14)

# Gaussian Basis Function
axs[1, 0].scatter(X_gaussian, y_gaussian, color='blue', alpha=0.5)
axs[1, 0].plot(X_gaussian, y_pred_gaussian, color='red')
axs[1, 0].set_title('C', fontsize=16, fontweight='bold')
axs[1, 0].set_xlabel('X', fontsize=14)
axs[1, 0].set_ylabel('y', fontsize=14)

# Menggambar contoh basis Fourier
axs[1, 1].scatter(X_fourier, y_fourier, color='blue', alpha=0.5)
axs[1, 1].plot(X_fourier, y_pred_fourier, color='red')
axs[1, 1].set_title('D', fontsize=16, fontweight='bold')
axs[1, 1].set_xlabel('X', fontsize=14)
axs[1, 1].set_ylabel('y', fontsize=14)

plt.tight_layout()
plt.show()
# # Menyimpan gambarnya
# fig.savefig('basis_functions.png', dpi=300)

In [ ]:
# Regresi linear biasa (data linear berderau)
X_lr = np.linspace(0, 10, 100).reshape(-1, 1)
y_lr = 2 * X_lr.flatten() + np.random.normal(0, 0.5, X_lr.shape[0])

# Basis polinomial (data kuadratik berderau)
X_poly = np.linspace(0, 10, 100).reshape(-1, 1)
y_poly = 0.5 * X_poly.flatten()**2 - 3 * X_poly.flatten() + 5 + np.random.normal(0, 2, X_poly.shape[0])

# Basis Gaussian (data dengan puncak terlokalisasi)
X_gaussian = np.linspace(0, 10, 100).reshape(-1, 1)
y_gaussian = np.exp(-0.5 * (X_gaussian.flatten() - 5)**2) + np.random.normal(0, 0.1, X_gaussian.shape[0])

# Basis Fourier (data berkala)
X_fourier = np.linspace(0, 10, 100).reshape(-1, 1)
y_fourier = np.sin(2 * np.pi * X_fourier.flatten() / 5) + np.random.normal(0, 0.1, X_fourier.shape[0])  # derau diperkecil

# Melatih regresi linear biasa
model_lr = LinearRegression()
model_lr.fit(X_lr, y_lr)
y_pred_lr = model_lr.predict(X_lr)

# Melatih model dengan basis polinomial
degree = 3
model_poly = make_pipeline(PolynomialFeatures(degree), LinearRegression())
model_poly.fit(X_poly, y_poly)
y_pred_poly = model_poly.predict(X_poly)

# Gaussian Basis Function
kernel = RBF(length_scale=1.0)
model_gaussian = GaussianProcessRegressor(kernel=kernel, alpha=0.1)
model_gaussian.fit(X_gaussian, y_gaussian)
y_pred_gaussian, _ = model_gaussian.predict(X_gaussian, return_std=True)

# Fourier Basis Function
def fourier_basis(x, n_terms=4):
    basis = [np.ones_like(x)]
    for i in range(1, n_terms + 1):
        basis.append(np.sin(i * x))
        basis.append(np.cos(i * x))
    return np.column_stack(basis)

X_fourier_basis = fourier_basis(X_fourier.flatten(), n_terms=4)
model_fourier = LinearRegression()
model_fourier.fit(X_fourier_basis, y_fourier)
y_pred_fourier = model_fourier.predict(X_fourier_basis)

fig, axs = plt.subplots(2, 2, figsize=(12, 10))

# Regular Linear Regression
axs[0, 0].scatter(X_lr, y_lr, color='blue', alpha=0.5, label='Data')
axs[0, 0].plot(X_lr, y_pred_lr, color='red', label='Regresi linear')
axs[0, 0].set_title('Regresi Linear Biasa', fontsize=16, fontweight='bold')
axs[0, 0].set_xlabel('X', fontsize=14)
axs[0, 0].set_ylabel('y', fontsize=14)
axs[0, 0].legend()

# Polynomial Basis Function
axs[0, 1].scatter(X_poly, y_poly, color='blue', alpha=0.5, label='Data')
axs[0, 1].plot(X_poly, y_pred_poly, color='red', label=f'Polinomial (derajat={degree})')
axs[0, 1].set_title('Basis Polinomial', fontsize=16, fontweight='bold')
axs[0, 1].set_xlabel('X', fontsize=14)
axs[0, 1].set_ylabel('y', fontsize=14)
axs[0, 1].legend()

# Gaussian Basis Function
axs[1, 0].scatter(X_gaussian, y_gaussian, color='blue', alpha=0.5, label='Data')
axs[1, 0].plot(X_gaussian, y_pred_gaussian, color='red', label='Basis Gaussian')
axs[1, 0].set_title('Basis Gaussian (RBF)', fontsize=16, fontweight='bold')
axs[1, 0].set_xlabel('X', fontsize=14)
axs[1, 0].set_ylabel('y', fontsize=14)
axs[1, 0].legend()

# Menggambar contoh basis Fourier
axs[1, 1].scatter(X_fourier, y_fourier, color='blue', alpha=0.5, label='Data')
axs[1, 1].plot(X_fourier, y_pred_fourier, color='red', label='Basis Fourier')
axs[1, 1].set_title('Basis Fourier', fontsize=16, fontweight='bold')
axs[1, 1].set_xlabel('X', fontsize=14)
axs[1, 1].set_ylabel('y', fontsize=14)
axs[1, 1].legend()

plt.tight_layout()
plt.show()
# # Menyimpan gambarnya
# fig.savefig('basis_functions_labeled.png', dpi=300)

Keempat panel memperlihatkan hal yang sama: **fungsi basis yang cocok akan mengikuti pola datanya**, sedangkan yang tidak cocok akan meleset.

Perhatikan bahwa keempat model itu semuanya adalah **regresi linear** — yang berbeda hanya transformasi yang diterapkan pada masukannya sebelum regresi dijalankan.

In [ ]:
x = np.linspace(-1, 1, 400)[:, None]
x_flat = x[:, 0]

phi_poly = x_flat ** 3  # basis polinomial: x^3
phi_rbf = np.exp(-15.0 * x_flat**2)  # basis Gaussian (RBF), γ = 15, pusat = 0
phi_four = np.sin(3 * np.pi * x_flat)  # basis Fourier: sin(3πx)

plt.figure(figsize=(12, 8))

plt.plot(x_flat, phi_poly, color='darkblue', linestyle='-', linewidth=3, label=r"Polinomial: $\phi_{\mathrm{poly}}(x)=x^3$")
plt.plot(x_flat, phi_rbf, color='darkgreen', linestyle='--', linewidth=3, label=r"Gaussian (RBF): $\phi_{\mathrm{rbf}}(x)=e^{-15x^{2}}$")
plt.plot(x_flat, phi_four, color='darkorange', linestyle='-.', linewidth=3, label=r"Fourier: $\phi_{\mathrm{four}}(x)=\sin(3\pi x)$")

plt.text(0, 1.05, "Puncak RBF", ha="center", fontsize=14, color='darkgreen')
plt.text(-0.7, -0.9, "Polinomial ($x^3$)", ha="center", fontsize=14, color='darkblue')
plt.text(0.7, 0.9, r"Fourier ($\sin(3\pi x)$)", ha="center", fontsize=14, color='darkorange')

plt.title("Contoh Fungsi Basis dari Beberapa Keluarga", fontsize=18, fontweight='bold')
plt.xlabel("$x$", fontsize=16)
plt.ylabel(r"$\phi(x)$", fontsize=16)
plt.legend(fontsize=14, loc='upper right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.ylim([-1.2, 2])
plt.tight_layout()

# # Menyimpan dan menampilkan grafiknya
# plt.savefig('basis_functions_pedagogical.png', dpi=300)
plt.show()

Grafik ini membandingkan bentuk ketiga jenis fungsi basis pada rentang yang sama:

* **Polinomial $x^3$** (biru) bersifat **global**: nilainya terus membesar menjauhi pusat, sehingga memengaruhi seluruh rentang.
* **Gaussian RBF** (hijau) bersifat **lokal**: hanya bernilai berarti di sekitar pusatnya, dan cepat meluruh menjadi nol.
* **Fourier** (oranye) bersifat **berkala**: berulang dengan pola yang sama sepanjang sumbu.

Perbedaan sifat inilah yang menentukan basis mana yang cocok untuk suatu data.

---
## 3. Membandingkan Regresi Linear dan Polinomial

Regresi linear mengandaikan hubungan berupa **garis lurus** antara fitur masukan dan target. Namun andaian ini tidak selalu benar, terutama untuk data yang berpola non-linear. Regresi polinomial memperluasnya dengan menambahkan suku-suku berpangkat dari fitur masukan.

| | Regresi Linear | Regresi Polinomial |
|---|---|---|
| Bentuk | Garis lurus | Kurva melengkung |
| Kelebihan | Sederhana dan mudah ditafsirkan | Mampu memodelkan hubungan non-linear |
| Kelemahan | Berisiko **underfitting** pada data rumit | Berisiko **overfitting** bila derajatnya terlalu tinggi |

Pada sel berikut kita membandingkan kinerja kedua model pada data yang sama, yaitu data yang sebenarnya berpola $\sin(5x)$. Kita memakai **Mean Squared Error (MSE)** untuk menilai kecocokannya — semakin kecil MSE, semakin dekat prediksi dengan data.

> **Waspadai jebakannya.** MSE yang dihitung pada data yang sama dengan data pelatihan akan **selalu** membaik seiring naiknya derajat polinomial. Untuk menilai model dengan jujur, MSE harus dihitung pada data uji yang terpisah — seperti yang kita lakukan pada Pertemuan 03.


In [ ]:
# Data sebenarnya berpola sin(5x) — jelas tidak linear
X = np.random.rand(100, 1)*2 - 1
y = np.sin(5*X).ravel() + 0.1*np.random.randn(100)

# Melatih model regresi linear
lin = LinearRegression()
lin.fit(X, y)

def get_linear_equation(intercept, coefficient):
    eq = f'y(x) = {intercept:.3f}'
    if coefficient >= 0:
        eq += f' + {coefficient:.3f}x'
    else:
        eq += f' - {-coefficient:.3f}x'
    return f'${eq}$'

linear_equation = get_linear_equation(lin.intercept_, lin.coef_[0])

idx = np.argsort(X.ravel())
plt.figure(figsize=(10, 7))

plt.scatter(X, y, color='darkblue', alpha=0.7, label="Titik data", s=80)

from sklearn.metrics import mean_squared_error

# Menghitung MSE untuk model regresi linear
mse_lin = mean_squared_error(y, lin.predict(X))

# Garis regresi linear
plt.plot(X[idx], lin.predict(X)[idx], color='darkorange', linestyle='-', linewidth=3,
         label=f"Kecocokan linear (MSE={mse_lin:.3f})")

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

# Menyusun dan melatih model regresi polinomial
degree = 5  # derajat polinomialnya
lin_poly = make_pipeline(PolynomialFeatures(degree), LinearRegression())
lin_poly.fit(X, y)

# Membuat prediksi dengan model polinomial
poly = X
mse_poly = np.mean((y - lin_poly.predict(poly))**2)

# Kurva regresi polinomial
plt.plot(X[idx], lin_poly.predict(poly)[idx], color='darkgreen', linestyle='--', linewidth=3,
         label=f"Kecocokan polinomial (derajat={degree}, MSE={mse_poly:.3f})")

# Menyusun teks persamaan polinomialnya
def get_polynomial_equation(coefficients):
    terms = [f'{coefficients[0]:.3f}']
    for i, coef in enumerate(coefficients[1:], start=1):
        if coef >= 0:
            terms.append(f'+ {coef:.3f}x^{i}')
        else:
            terms.append(f'- {-coef:.3f}x^{i}')
    return f"$y(x) = {' '.join(terms)}$"

poly_equation = get_polynomial_equation(lin_poly.named_steps['linearregression'].coef_)

plt.text(-0.9, -1.8, poly_equation, fontsize=14, color='darkgreen', ha='left')
plt.text(-0.9, -2.2, linear_equation, fontsize=14, color='darkorange', ha='left')

plt.title("Regresi Linear vs Regresi Polinomial", fontsize=18, fontweight='bold')
plt.xlabel("x", fontsize=16)
plt.ylabel("y", fontsize=16)
plt.legend(fontsize=14, loc='lower right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()

# # Menyimpan dan menampilkan grafiknya
# plt.savefig('linear_vs_polynomial_pedagogical.png', dpi=300)
plt.show()


**Bandingkan kedua nilai MSE pada legenda.** Model polinomial jauh lebih kecil MSE-nya karena mampu mengikuti liukan data, sedangkan garis lurus jelas *underfitting* pada data berpola sinus ini.

> **Tetapi hati-hati.** MSE di sini dihitung pada data yang sama dengan data pelatihan. Menaikkan derajat polinomial akan **selalu** menurunkan angka ini, bahkan ketika modelnya sudah mulai menghafal derau. Penilaian yang jujur menuntut data uji terpisah.

---
## 4. Geometri Regresi Linear

Bagian ini menyajikan regresi linear dari sudut pandang **aljabar linear**. Sudut pandang ini menjelaskan *mengapa* rumus penyelesaian regresi linear berbentuk seperti yang kita kenal.

Pada sel berikutnya kita akan menampilkan:

* **Residual**: selisih antara nilai prediksi dan nilai target yang sebenarnya.
* **Rentang fitur masukan** (*span* dari $\mathbb{X}$): subruang yang dibentuk oleh semua kombinasi linear dari vektor-vektor kolom matriks fitur.

**Prediksi sebagai kombinasi linear**

Prediksi $\mathbf{Y}(\mathbb{X}, \mathbf{w}) = \mathbb{X}\mathbf{w}$ adalah **kombinasi linear dari kolom-kolom** $\mathbb{X}$. Artinya vektor prediksi $\mathbf{Y}$ selalu terletak di dalam **rentang** matriks masukan, ditulis $\text{span}(\mathbb{X})$.

**Tafsirannya**

* Vektor prediksi $\mathbf{Y}$ **selalu** berada di dalam $\text{span}(\mathbb{X})$, sekalipun vektor target sebenarnya $\mathbf{t}$ berada di luarnya.
* Tujuan regresi linear adalah menemukan vektor $\mathbf{Y}$ di dalam $\text{span}(\mathbb{X})$ yang **paling dekat** dengan $\mathbf{t}$.

**Meminimalkan residual**

* Vektor residual $\mathbf{e} = \mathbf{t} - \mathbf{Y}$ menyatakan selisih antara target sebenarnya dan prediksi.
* Meminimalkan jarak antara $\mathbf{Y}$ dan $\mathbf{t}$ berarti meminimalkan **panjang** vektor residual $\mathbf{e}$.

**Proyeksi ortogonal**

* Panjang $\mathbf{e}$ menjadi minimum tepat ketika $\mathbf{Y}$ merupakan **proyeksi ortogonal** dari $\mathbf{t}$ ke $\text{span}(\mathbb{X})$.
* Dengan demikian $\mathbf{e}$ tegak lurus terhadap $\text{span}(\mathbb{X})$ — inilah syarat kecocokan terbaik pada regresi linear.

> **Analogi sederhana.** Bayangkan $\text{span}(\mathbb{X})$ sebagai lantai sebuah ruangan, dan $\mathbf{t}$ sebagai sebuah titik yang melayang di udara. Titik di lantai yang paling dekat dengan titik melayang itu adalah **bayangannya tepat di bawahnya**, dan garis penghubungnya tegak lurus terhadap lantai. Bayangan itulah $\mathbf{Y}$, dan garis tegak lurusnya adalah residual $\mathbf{e}$.

Grafik tiga dimensi berikut menggambarkan hal itu. Putar grafiknya untuk melihat bahwa panah oranye (residual) benar-benar tegak lurus terhadap bidangnya.


In [ ]:
# ============ Parameter tampilan ============
SCALE        = 2.0
LINE_W       = 14
HEAD_SIZE_T  = 1.9
HEAD_SIZE_E  = 1.5
COEFF_RANGE  = 3.0
GRID_LINES   = 11
PAD_RATIO    = 0.20
BRACKET_FRAC = 0.22
# =================================================

x1 = np.array([3.0, 0.4, 0.6]) * SCALE
x2 = np.array([1.2, 2.4, -0.5]) * SCALE

w1, w2 = 1.5, 0.9
residual_height = 2.2 * SCALE

O = np.zeros(3)
Y = w1*x1 + w2*x2

def unit(v):
    n = np.linalg.norm(v);  return v/n if n else v

def gram_schmidt(a, b):
    u1 = unit(a)
    b_perp = b - np.dot(b, u1)*u1
    u2 = unit(b_perp)
    return u1, u2

u_hat, v_hat = gram_schmidt(x1, x2)
n_hat = unit(np.cross(u_hat, v_hat))

t = Y + residual_height*n_hat

nu = nv = 45
U = np.linspace(-COEFF_RANGE, COEFF_RANGE, nu)
V = np.linspace(-COEFF_RANGE, COEFF_RANGE, nv)
UU, VV = np.meshgrid(U, V)
P = UU[...,None]*x1 + VV[...,None]*x2

plane = go.Surface(
    x=P[...,0], y=P[...,1], z=P[...,2],
    opacity=0.22, showscale=False, name="span(X)",
    surfacecolor=np.zeros_like(P[...,0]),
    colorscale=[[0,"green"],[1,"green"]],
    hoverinfo="skip"
)

grid = []
for s in np.linspace(-COEFF_RANGE, COEFF_RANGE, GRID_LINES):
    a, b = s*x1 + (-COEFF_RANGE)*x2, s*x1 + (COEFF_RANGE)*x2
    grid.append(go.Scatter3d(x=[a[0],b[0]], y=[a[1],b[1]], z=[a[2],b[2]],
                             mode="lines", line=dict(width=2, color="rgba(0,90,0,0.6)"),
                             showlegend=False, hoverinfo="skip"))
    a, b = (-COEFF_RANGE)*x1 + s*x2, (COEFF_RANGE)*x1 + s*x2
    grid.append(go.Scatter3d(x=[a[0],b[0]], y=[a[1],b[1]], z=[a[2],b[2]],
                             mode="lines", line=dict(width=2, color="rgba(0,90,0,0.6)"),
                             showlegend=False, hoverinfo="skip"))


line_traces, cone_traces = [], []

def arrow(start, end, name, color, width=LINE_W, head=1.6, back=None, dash=None, show_legend=True):
    vec = end - start
    L = np.linalg.norm(vec)
    if L == 0: return
    if back is None:
        back = max(0.9*head, 0.02*L)
    tip_base = end - back*(vec/L)

    line_traces.append(go.Scatter3d(
        x=[start[0], tip_base[0]], y=[start[1], tip_base[1]], z=[start[2], tip_base[2]],
        mode="lines",
        line=dict(width=width, color=color, dash=dash) if dash else dict(width=width, color=color),
        name=name, showlegend=show_legend, hoverinfo="skip"
    ))
    cone_traces.append(go.Cone(
        x=[end[0]], y=[end[1]], z=[end[2]],
        u=[vec[0]], v=[vec[1]], w=[vec[2]],
        sizemode="absolute", sizeref=head, anchor="tip",
        showscale=False, colorscale=[[0, color],[1, color]],
        name=name, showlegend=False
    ))

# vektor-vektor kolom (basis dari span)
arrow(O, x1, "X·,1", "darkgreen")
arrow(O, x2, "X·,2", "darkgreen")

# Y (vektor prediksi)
arrow(O, Y, "Y = Xw", "black")

# t
arrow(O, t, "t", "#d62728", head=HEAD_SIZE_T)

# vektor residual e
arrow(Y, t, "e = t − Xw", "#ff9800", head=HEAD_SIZE_E, width=LINE_W-2)

# Penanda sudut siku-siku (bukti ketegaklurusan)
y_dir = unit(Y - O)
e_dir = unit(t - Y)

tick  = BRACKET_FRAC * min(np.linalg.norm(x1), np.linalg.norm(x2)) * SCALE
h = tick

base   = Y - tick * y_dir
corner = base + h * e_dir
end_on_residual = Y + h * e_dir

bracket = go.Scatter3d(
    x=[base[0], corner[0], end_on_residual[0]],
    y=[base[1], corner[1], end_on_residual[1]],
    z=[base[2], corner[2], end_on_residual[2]],
    mode="lines",
    line=dict(width=10, color="royalblue"),
    showlegend=False, hoverinfo="skip"
)

p1 = Y - tick * y_dir
p2 = p1 + tick * e_dir

pts = np.vstack([O, x1, x2, Y, t, p1, p2])
mins, maxs = pts.min(axis=0), pts.max(axis=0)
pad = PAD_RATIO * np.max(maxs - mins if np.any(maxs - mins) else np.array([1,1,1]))
xr, yr, zr = [mins[0]-pad, maxs[0]+pad], [mins[1]-pad, maxs[1]+pad], [mins[2]-pad, maxs[2]+pad]

# label pada gambar
def label(pt, text, d=0.12*SCALE):
    return dict(x=pt[0]+d, y=pt[1]+d, z=pt[2]+d, text=text,
                showarrow=False, bgcolor="rgba(255,255,255,0.85)", bordercolor="black")
ann = [label(x1,"X<sub>·,1</sub>"), label(x2,"X<sub>·,2</sub>"),
       label(Y,"Y = Xw"), label(t,"t"), label((Y+t)/2,"e")]

fig = go.Figure(data=[plane, *grid, bracket, *line_traces, *cone_traces])  # kerucut digambar terakhir

fig.update_layout(
    title="Geometri Regresi Linear: Proyeksi Ortogonal",
    width=1100, height=900,
    scene=dict(
        xaxis=dict(title="x", range=xr, zeroline=False, showbackground=False),
        yaxis=dict(title="y", range=xr, zeroline=False, showbackground=False),
        zaxis=dict(title="z", range=xr, zeroline=False, showbackground=False),
        annotations=ann,
        aspectmode="cube",
        camera=dict(eye=dict(x=1, y=-1, z=1.4))
    ),
    legend=dict(x=0.02, y=0.98, bgcolor="rgba(255,255,255,0.75)")
)

fig.show()
# fig.write_image("linreg-3d-geometry.png", scale = 4)
# fig.write_html("linreg-3d-geometry.html", include_plotlyjs='cdn')

Bacalah gambar di atas seperti berikut:

* Dua panah **hijau** adalah vektor kolom matriks $\mathbb{X}$. Keduanya membentangkan sebuah bidang, yaitu $\text{span}(\mathbb{X})$.
* Panah **hitam** adalah prediksi $\mathbf{Y} = \mathbb{X}\mathbf{w}$, yang selalu terletak **di dalam** bidang itu.
* Panah **merah** adalah target sebenarnya $\mathbf{t}$, yang umumnya berada **di luar** bidang.
* Panah **oranye** adalah residual $\mathbf{e} = \mathbf{t} - \mathbf{Y}$, dan penanda siku-siku biru menunjukkan bahwa panah ini **tegak lurus** terhadap bidangnya.

> **Inilah inti regresi linear kuadrat terkecil.** Karena $\mathbf{t}$ pada umumnya tidak dapat dicapai secara persis, kita mengambil titik terdekat yang bisa dicapai, yaitu proyeksi ortogonalnya. Syarat ketegaklurusan $\mathbb{X}^T\mathbf{e} = 0$ inilah yang bila diuraikan menghasilkan persamaan normal $\mathbf{w} = (\mathbb{X}^T\mathbb{X})^{-1}\mathbb{X}^T\mathbf{t}$.

---
## 5. Menganalisis Residual

**Residual** adalah selisih antara nilai yang teramati (target sebenarnya) dan nilai prediksi model. Menganalisis residual membantu kita menilai kecocokan model sekaligus mengenali masalah seperti hubungan non-linear, heteroskedastisitas, atau pencilan.

**1. Plot residual**

* Grafik pencar antara residual (sumbu-y) dan nilai prediksi (sumbu-x).
* Idealnya residual **tersebar acak di sekitar nol** tanpa pola apa pun. Pola seperti itu menandakan model sudah menangkap seluruh struktur data.
* Bila terlihat pola melengkung, berarti masih ada hubungan non-linear yang belum tertangkap model.

**2. Heteroskedastisitas**

* Terjadi ketika **ragam residual tidak tetap** di sepanjang rentang nilai prediksi.
* Sering tampak sebagai pola **menyerupai kipas** (menyebar melebar) pada plot residual.
* Menandakan bahwa ketidakpastian prediksi berbeda-beda pada tiap rentang nilai — misalnya harga rumah mahal jauh lebih sulit diprediksi daripada rumah murah.

> **Mengapa penting?** Banyak uji statistik pada regresi mengandaikan ragam residual yang tetap (*homoskedastisitas*). Bila andaian itu dilanggar, prediksinya masih boleh dipakai, tetapi ukuran ketidakpastiannya menjadi tidak dapat dipercaya.

---
## 6. Menerapkan Regresi Linear pada Data Nyata

Pada sel berikut kita memakai dataset **California Housing** untuk melatih model regresi linear sederhana: memprediksi **nilai tengah harga rumah** dari **pendapatan tengah** penduduk suatu blok.

Kita mengevaluasinya dengan tiga metrik:

| Metrik | Arti | Nilai yang baik |
|---|---|---|
| **MSE** | Rata-rata kuadrat galat prediksi | Sekecil mungkin |
| **R²** | Proporsi ragam target yang dapat dijelaskan model | Mendekati 1 |
| **R² terkoreksi** | R² yang dikoreksi terhadap banyaknya fitur | Mendekati 1 |

Perhatikan bahwa metrik dihitung pada **data uji**, bukan data latih — sesuai kaidah yang sudah kita pelajari pada Pertemuan 03.


In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd

data = fetch_california_housing(as_frame=True)
df = data.frame.copy()

df = df[["MedInc", "MedHouseVal"]].rename(columns={"MedHouseVal": "y"})

# Menyaring data untuk membuang pencilan ekstrem
df = df[(df["MedInc"] < 10) & (df["y"] < 5)]

# Sampel berstrata menurut kuantil MedInc: cakupan tetap luas, tampilan tidak terlalu padat
np.random.seed(7)
q = pd.qcut(df["MedInc"], q=15, duplicates="drop")
sampled = (
    df.groupby(q, observed=True)
      .apply(lambda g: g.sample(n=min(len(g), 30), random_state=7))
      .reset_index(drop=True)
)

X = sampled[["MedInc"]].values
y = sampled["y"].values

# Membagi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Melatih model regresi linear
model = LinearRegression()
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test  = model.predict(X_test)

resid_train = y_train - y_pred_train
resid_test  = y_test  - y_pred_test

# Menghitung metrik pada data uji
mse = mean_squared_error(y_test, y_pred_test)
r2  = r2_score(y_test, y_pred_test)
n, p = X_test.shape[0], X_test.shape[1]
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print(f"MSE data uji      : {mse:.4f}")
print(f"R-kuadrat data uji: {r2:.4f}")
print(f"R-kuadrat terkoreksi: {adj_r2:.4f}")

def binned_trend(x, y, bins=12):
    edges = np.linspace(np.min(x), np.max(x), bins + 1)
    idx = np.digitize(x, edges) - 1
    xc = []
    yc = []
    for b in range(bins):
        mask = idx == b
        if np.any(mask):
            xc.append(np.mean(x[mask]))
            yc.append(np.mean(y[mask]))
    return np.array(xc), np.array(yc)

# Grafik 1: data dan garis hasil pelatihan
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
axs[0].scatter(X_train, y_train, s=50, alpha=0.7, edgecolor="k", color="crimson")
xline = np.linspace(X_train.min(), X_train.max(), 200).reshape(-1, 1)
yline = model.predict(xline)
axs[0].plot(xline, yline, lw=3, label="Garis hasil pelatihan", color="blue", zorder=3)
axs[0].set_xlabel("Pendapatan tengah (MedInc)", fontsize=14)
axs[0].set_ylabel("Nilai tengah harga rumah (y)", fontsize=14)
axs[0].set_title("Regresi Linear Sederhana: y ~ MedInc", fontsize=16, fontweight="bold")
axs[0].legend(frameon=False, fontsize=12)
axs[0].grid(True, linestyle="--", alpha=0.6)

# Grafik 2: residual terhadap nilai prediksi
axs[1].scatter(y_pred_train, resid_train, s=50, alpha=0.7, edgecolor="k", color="crimson")
axs[1].axhline(0, color="k", lw=2, ls="--", label="Garis residual nol")
axs[1].set_xlabel("Nilai prediksi", fontsize=14)
axs[1].set_ylabel("Residual", fontsize=14)
axs[1].set_title("Residual vs Nilai Prediksi\nPola kipas → heteroskedastisitas", fontsize=16, fontweight="bold")
axs[1].legend(frameon=False, fontsize=12)
axs[1].grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()
# fig.savefig("linreg-california-housing.png", dpi=300)

**Bacalah kedua grafik.**

Grafik kiri menunjukkan garis regresi menembus awan titik data. Kecenderungan naik terlihat jelas: semakin tinggi pendapatan, semakin tinggi harga rumah.

Grafik kanan adalah **plot residual**. Perhatikan bentuknya yang melebar seperti kipas ke arah kanan — inilah **heteroskedastisitas**. Artinya, prediksi model jauh lebih tidak pasti pada daerah berpendapatan tinggi dibandingkan daerah berpendapatan rendah.

Perhatikan pula nilai R² yang tercetak. Nilai itu memberi tahu berapa besar bagian ragam harga rumah yang berhasil dijelaskan oleh pendapatan saja. Sisanya berasal dari faktor lain yang belum masuk ke dalam model — lokasi, usia bangunan, jumlah kamar, dan sebagainya.

---
## 7. Penutup

### Ringkasan

**Regresi linear**

* Model regresi linear berbentuk $y(x,w) = w_0 + w_1 x_1 + \dots + w_D x_D$, dengan $w_0$ sebagai intersep dan $w_j$ sebagai bobot tiap fitur.
* Disebut "linear" karena **linear terhadap parameter $w$**, bukan karena grafiknya harus lurus.
* Dengan satu fitur prediksinya berupa garis; dengan dua fitur berupa bidang; dengan $D$ fitur berupa hiperbidang.

**Fungsi basis**

* Fungsi basis $\phi_j(x)$ mentransformasi masukan sehingga regresi linear mampu memodelkan pola non-linear.
* Empat keluarga yang umum: polinomial (global), RBF/Gaussian (lokal), sigmoid (peralihan), dan Fourier (berkala).
* Memilih fungsi basis sama dengan menanamkan bias induktif tentang bentuk pola datanya.

**Kompleksitas model**

* Derajat polinomial yang terlalu rendah menyebabkan *underfitting*, terlalu tinggi menyebabkan *overfitting*.
* MSE pada data latih selalu membaik seiring naiknya kompleksitas, sehingga penilaian harus dilakukan pada data uji.

**Geometri**

* Prediksi $\mathbf{Y} = \mathbb{X}\mathbf{w}$ selalu terletak di dalam $\text{span}(\mathbb{X})$.
* Solusi terbaik adalah **proyeksi ortogonal** dari $\mathbf{t}$ ke $\text{span}(\mathbb{X})$, sehingga residual $\mathbf{e}$ tegak lurus terhadap seluruh kolom $\mathbb{X}$.

**Residual**

* Residual yang tersebar acak di sekitar nol menandakan model sudah cocok.
* Pola menyerupai kipas menandakan **heteroskedastisitas**, yaitu ragam galat yang tidak tetap.

### Latihan Mandiri

1. Pada sel pertama, ubah besar derau dari `np.random.randn(N) * 2` menjadi `* 0.5` lalu `* 6`. Bagaimana pengaruhnya terhadap tampilan sebaran titik di sekitar garis?
2. Pada perbandingan regresi linear dan polinomial, ubah `degree` menjadi 1, 3, 5, 10, dan 20. Pada derajat berapa kurvanya mulai terlihat "meliuk berlebihan" mengikuti derau, bukan mengikuti pola sebenarnya?
3. Bagi data pada latihan nomor 2 menjadi data latih dan data uji, lalu hitung MSE keduanya untuk setiap derajat. Buat grafik MSE terhadap derajat. Di mana letak titik seimbangnya?
4. Data pada bagian 3 sebenarnya berpola $\sin(5x)$. Menurut Anda, fungsi basis mana yang paling tepat untuk data ini: polinomial, RBF, atau Fourier? Ujilah dugaan Anda.
5. Pada grafik geometri tiga dimensi, putar tampilannya sampai Anda melihat bidangnya dari samping. Buktikan secara numerik bahwa residual tegak lurus terhadap kedua vektor kolom, yaitu dengan memeriksa bahwa hasil kali titiknya mendekati nol.
6. Pada bagian California Housing, tambahkan fitur kedua (misalnya `AveRooms`) ke dalam model. Apakah R² meningkat? Apakah pola kipas pada plot residual berkurang?
7. Jelaskan dengan kalimat Anda sendiri: mengapa pola kipas pada plot residual menjadi masalah, padahal prediksinya sendiri boleh jadi masih cukup baik?

### Bacaan Lanjutan

* Dokumentasi regresi linear scikit-learn: https://scikit-learn.org/stable/modules/linear_model.html
* Transformasi fitur polinomial: https://scikit-learn.org/stable/modules/preprocessing.html#polynomial-features
